# Tensor-v2 encoder comparison on Kaggle

## Experiment goal

This notebook uses one Kaggle GPU session for the two focused follow-ups to the
first long run:

1. a three-layer, **one-head heterogeneous GATv2**;
2. a matched three-layer **non-attention relational encoder**.

The models share the input projection, multi-statistic readout, global shortcut,
prediction heads, loss, split, seed, and optimizer settings. The intended
comparison therefore isolates attention versus relation-specific message passing;
the one-head result also tests whether the original multi-head attention was
unnecessary overhead.

The default caps reserve 390 minutes for GATv2 and 230 minutes for the faster
relational model: 10 hours 20 minutes of training within an 11-hour allowance.
The remaining time is for setup and best-checkpoint evaluation. Early stopping
can finish either model sooner.

Both experiments are resumable. The trainer writes the best checkpoint whenever
validation SMAPE improves and an optimizer backup every five completed epochs.
Re-running a cell resumes its local backup; a later Kaggle session can import the
packaged results by setting `PREVIOUS_RESULTS_ROOT`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import torch

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "ll-hls4ml"
RESULTS_DIR = WORK / "results"
CONFIG_DIR = WORK / "configs"

REPO_URL = "https://github.com/brios-polimi/ll-hls4ml.git"
# This commit introduces the shared input projection and relational encoder.
REPO_REF = "ad047431dc2dacacbb4fbe170b10e00bcd8ba4ad"

TENSOR_REPO_ID = "BrendanRios/wa-hls4ml-tensors-pragmas-v2"
TENSOR_REVISION = "8abfe2314ef9144eb8fd612d30babfcfca173c51"
TENSOR_DIR = WORK / "tensors_v2"

# To resume in another Kaggle session, attach the prior result zip as an input
# and point this at the attached directory containing it (or extracted results).
PREVIOUS_RESULTS_ROOT = None

SEED = 42
RUN_GATV2_1HEAD = True
RUN_RELATIONAL = True
GATV2_TRAIN_BUDGET = "390m"
RELATIONAL_TRAIN_BUDGET = "230m"
ARCHIVE_NAME = "ll_hls4ml_encoder_comparison_results"

GPU_COUNT = torch.cuda.device_count()
assert GPU_COUNT >= 1, "Enable a Kaggle GPU accelerator before running."
print("PyTorch:", torch.__version__)
print("CUDA devices:", GPU_COUNT)
for index in range(GPU_COUNT):
    print(index, torch.cuda.get_device_name(index))
print("Training caps:", GATV2_TRAIN_BUDGET, "+", RELATIONAL_TRAIN_BUDGET)


## Install dependencies and checkout the pinned training code

In [ ]:
# Kaggle already supplies CUDA-enabled PyTorch.
%pip install -q torch-geometric huggingface_hub pyyaml pandas matplotlib


In [ ]:
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags"],
        check=True,
    )
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    ["git", "-C", str(REPO_DIR), "checkout", REPO_REF],
    check=True,
)
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()
assert commit == REPO_REF, (commit, REPO_REF)
print("ll-hls4ml commit:", commit)

sys.path.insert(0, str(REPO_DIR / "src"))
from ll_hls4ml.models.registry import list_models

available_models = list_models()
assert "hetero_gat" in available_models
assert "hetero_relational" in available_models
print("Available models:", available_models)


## Download and validate tensor-v2

In [ ]:
from huggingface_hub import login, snapshot_download

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)

print("Downloading", TENSOR_REPO_ID, "revision", TENSOR_REVISION or "main")
snapshot_download(
    repo_id=TENSOR_REPO_ID,
    repo_type="dataset",
    revision=TENSOR_REVISION,
    token=hf_token,
    local_dir=str(TENSOR_DIR),
    allow_patterns=["*.pt", "*.json"],
)
print("Tensor snapshot downloaded to", TENSOR_DIR)


In [ ]:
import json
import torch

from ll_hls4ml.io.schema import (
    BLOCK_FEATURE_SIZE,
    GRAPH_CONTEXT_CATEGORICAL_VOCABS,
    GRAPH_CONTEXT_NUMERIC_KEYS,
    PRAGMA_FEATURE_SIZE,
)

tensor_files = sorted(TENSOR_DIR.rglob("*.pt"))
assert tensor_files, f"No .pt tensors found under {TENSOR_DIR}"

labels_path = TENSOR_DIR / "labels.json"
assert labels_path.is_file(), "tensor-v2 labels.json is missing"
labels_payload = json.loads(labels_path.read_text())
label_count = len(labels_payload.get("labels", {}))

sample = torch.load(tensor_files[0], map_location="cpu", weights_only=False)
errors = []
if sample["pragma"].x.shape[1] != PRAGMA_FEATURE_SIZE:
    errors.append(
        f"pragma width {sample['pragma'].x.shape[1]} != v2 width "
        f"{PRAGMA_FEATURE_SIZE}"
    )
if sample["block"].x.shape[1] != BLOCK_FEATURE_SIZE:
    errors.append(
        f"block width {sample['block'].x.shape[1]} != {BLOCK_FEATURE_SIZE}"
    )
if not hasattr(sample, "graph_context_categorical"):
    errors.append("graph_context_categorical is absent")
elif sample.graph_context_categorical.shape[-1] != len(
    GRAPH_CONTEXT_CATEGORICAL_VOCABS
):
    errors.append("categorical synthesis-context width is incorrect")
if not hasattr(sample, "graph_context_numeric"):
    errors.append("graph_context_numeric is absent")
elif sample.graph_context_numeric.shape[-1] != len(
    GRAPH_CONTEXT_NUMERIC_KEYS
):
    errors.append("numeric synthesis-context width is incorrect")

assert not errors, (
    "Downloaded tensors are not tensor-v2:\n- " + "\n- ".join(errors)
)

vocab_candidates = [
    TENSOR_DIR / "vocab.json",
    REPO_DIR / "artifacts" / "vocab" / "vocab.json",
]
VOCAB_PATH = next((path for path in vocab_candidates if path.is_file()), None)
assert VOCAB_PATH is not None, (
    "vocab.json is missing. Include the tensor-v2 build vocabulary in the "
    "Hub snapshot."
)

print("Tensor files:", len(tensor_files))
print("Indexed labels:", label_count)
print("Sample:", tensor_files[0].relative_to(TENSOR_DIR))
print("Pragma width:", sample["pragma"].x.shape[1])
print("Block width:", sample["block"].x.shape[1])
print(
    "Context widths:",
    sample.graph_context_categorical.shape[-1],
    sample.graph_context_numeric.shape[-1],
)
print("Vocabulary:", VOCAB_PATH)


## Matched long-run configurations

Both models use the official split, empirical distributed sampling,
multi-statistic count-aware pooling, the global feature shortcut, split
resource/timing towers, DSP/BRAM hurdle outputs, and robust regression in
`log1p(target)` space. Synthesis context remains disabled so this stays directly
comparable to the first long run.

`batch_size=1` is per GPU, giving a global batch of two on a dual-GPU Kaggle
instance. `patience=30` is intentionally retained: the earlier GAT curve had a
late improvement after roughly 29 non-improving epochs, so shortening patience
would introduce a meaningful risk and weaken comparability. The wall-clock caps,
not patience, protect the session budget.


In [ ]:
import json
import shutil

KERNEL_TYPES = [
    "2layer",
    "3layer",
    "conv1d",
    "conv2d",
    "dense_latency",
    "dense_resource",
    "rule4ml",
]

common = {
    "tensor_dir": str(TENSOR_DIR),
    "vocab_path": str(VOCAB_PATH),
    "results_dir": str(RESULTS_DIR),
    "kernel_types": KERNEL_TYPES,
    "seed": SEED,
    "split_strategy": "official_or_stratified",
    # WeightedRandomSampler is not sharded by the current DDP loader.
    "family_balanced_sampling": False,
    "batch_size": 1,
    "num_workers": 0,
    "epochs": 400,
    "patience": 30,
    "learning_rate": 5e-4,
    "weight_decay": 1e-4,
    "hidden_dim": 64,
    "num_layers": 3,
    "dropout": 0.15,
    "pool": "multi",
    "aggr": "sum",
    "use_global_features": True,
    "use_context": False,
    "split_heads": True,
    "hurdle_heads": True,
    "hurdle_prediction_mode": "threshold",
    "loss": "log_huber_hurdle",
    "log_huber_delta": 0.35,
    "hurdle_classification_weight": 0.25,
    "early_stopping_metric": "smape",
    "verbose": 2,
}

gatv2_config = {
    **common,
    "experiment_name": "kaggle_v2_gatv2_1head_3layer_seed42",
    "model": "hetero_gat",
    "heads": 1,
}

relational_config = {
    **common,
    "experiment_name": "kaggle_v2_relational_mean_3layer_seed42",
    "model": "hetero_relational",
    # Mean within each relation; sum across relation types.
    "message_aggr": "mean",
}

for config in (gatv2_config, relational_config):
    experiment = config["experiment_name"]
    config["checkpoint_dir"] = str(
        RESULTS_DIR / experiment / "checkpoints"
    )

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
_cached_previous_root = None


def previous_search_root():
    global _cached_previous_root
    if PREVIOUS_RESULTS_ROOT is None:
        return None
    if _cached_previous_root is not None:
        return _cached_previous_root

    root = Path(PREVIOUS_RESULTS_ROOT)
    archives = sorted(root.rglob(f"{ARCHIVE_NAME}.zip"))
    if archives:
        if len(archives) > 1:
            raise RuntimeError(f"Multiple previous result archives: {archives}")
        extracted = WORK / "previous_encoder_comparison_results"
        if not extracted.is_dir():
            shutil.unpack_archive(archives[0], extracted)
            print("Extracted previous results:", archives[0])
        root = extracted
    _cached_previous_root = root
    return root


def find_previous(name):
    root = previous_search_root()
    if root is None:
        return None
    matches = sorted(root.rglob(name))
    if len(matches) > 1:
        raise RuntimeError(f"Multiple resume candidates for {name}: {matches}")
    return matches[0] if matches else None


def write_config(config):
    path = CONFIG_DIR / f"{config['experiment_name']}.json"
    checkpoint_dir = Path(config["checkpoint_dir"])
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    backup_name = f"{config['experiment_name']}_backup.pt"
    best_name = f"{config['experiment_name']}_checkpoint.pt"
    backup = checkpoint_dir / backup_name
    best = checkpoint_dir / best_name

    previous_backup = find_previous(backup_name)
    previous_best = find_previous(best_name)
    if not backup.is_file() and previous_backup is not None:
        shutil.copy2(previous_backup, backup)
        print("Imported optimizer backup:", previous_backup)
    if not best.is_file() and previous_best is not None:
        shutil.copy2(previous_best, best)
        print("Imported best checkpoint:", previous_best)

    payload = dict(config)
    if backup.is_file():
        payload["resume_checkpoint_path"] = str(backup)
        print("Will resume", config["experiment_name"], "from", backup)
    path.write_text(json.dumps(payload, indent=2))
    return path


GATV2_CONFIG_PATH = write_config(gatv2_config)
RELATIONAL_CONFIG_PATH = write_config(relational_config)
print(GATV2_CONFIG_PATH)
print(RELATIONAL_CONFIG_PATH)


## Time-bounded training, evaluation, and incremental packaging

In [ ]:
import shlex
import subprocess
import time

TRAIN_SCRIPT = REPO_DIR / "scripts" / "train.py"
run_environment = os.environ.copy()
run_environment["PYTHONPATH"] = str(REPO_DIR / "src")
run_environment["LL_HLS4ML_TQDM"] = "0"
run_environment["MPLCONFIGDIR"] = "/kaggle/working/matplotlib"


def base_training_command(config_path):
    if GPU_COUNT > 1:
        return [
            sys.executable,
            "-m",
            "torch.distributed.run",
            "--standalone",
            f"--nproc_per_node={GPU_COUNT}",
            str(TRAIN_SCRIPT),
            "--config",
            str(config_path),
        ]
    return [
        sys.executable,
        str(TRAIN_SCRIPT),
        "--config",
        str(config_path),
    ]


def run_and_stream(command, log_path):
    print("$", shlex.join(command), flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", buffering=1) as log:
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            env=run_environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        return process.wait()


def package_results():
    archive = Path(
        shutil.make_archive(
            str(WORK / ARCHIVE_NAME),
            "zip",
            root_dir=RESULTS_DIR,
        )
    )
    print("Updated result archive:", archive)
    return archive


def run_experiment(config, budget):
    config_path = write_config(config)
    experiment = config["experiment_name"]
    run_dir = RESULTS_DIR / experiment
    summary_path = run_dir / "summary.json"

    if summary_path.is_file():
        existing = json.loads(summary_path.read_text())
        evaluation_checkpoint = existing.get("resolved_config", {}).get(
            "evaluation_checkpoint_path"
        )
        if evaluation_checkpoint is None:
            print(experiment, "already completed normally; skipping.")
            package_results()
            return
        print(experiment, "has a timeout evaluation; resuming training.")

    log_path = run_dir / "training.log"
    command = [
        "timeout",
        "--signal=INT",
        "--kill-after=5m",
        budget,
        *base_training_command(config_path),
    ]

    try:
        started = time.time()
        return_code = run_and_stream(command, log_path)
        elapsed = time.time() - started
        print(experiment, "training return code:", return_code)
        print(experiment, "training wall seconds:", round(elapsed, 1))

        if return_code == 0 and summary_path.is_file():
            print("Training completed normally; evaluation is already complete.")
            return

        best_checkpoint = (
            Path(config["checkpoint_dir"])
            / f"{experiment}_checkpoint.pt"
        )
        assert best_checkpoint.is_file(), (
            f"No best checkpoint exists for {experiment}: {best_checkpoint}"
        )
        print("Evaluating best checkpoint:", best_checkpoint)
        evaluation_command = [
            sys.executable,
            str(TRAIN_SCRIPT),
            "--config",
            str(config_path),
            "--evaluate-checkpoint",
            str(best_checkpoint),
        ]
        evaluation_code = run_and_stream(
            evaluation_command,
            run_dir / "evaluation.log",
        )
        assert evaluation_code == 0, (
            f"Best-checkpoint evaluation failed with code {evaluation_code}"
        )
    finally:
        # Preserve the first result before beginning the second long run, and
        # preserve checkpoints/logs even if evaluation raises an error.
        package_results()


## Run 1 — one-head GATv2

This is the direct head-count ablation. It runs first because its attention
layers are expected to be slower; the result archive is refreshed before the
relational experiment begins.


In [ ]:
if RUN_GATV2_1HEAD:
    run_experiment(gatv2_config, GATV2_TRAIN_BUDGET)
else:
    print("Skipping one-head GATv2 by configuration.")


## Run 2 — non-attention relational encoder

This model keeps relation-specific transformations and edge-position features,
but replaces attention with mean aggregation within each relation.


In [ ]:
if RUN_RELATIONAL:
    run_experiment(relational_config, RELATIONAL_TRAIN_BUDGET)
else:
    print("Skipping relational encoder by configuration.")


## Compare and download results

In [ ]:
import pandas as pd

rows = []
for config, enabled in (
    (gatv2_config, RUN_GATV2_1HEAD),
    (relational_config, RUN_RELATIONAL),
):
    if not enabled:
        continue
    run_dir = RESULTS_DIR / config["experiment_name"]
    metrics_path = run_dir / "metrics.csv"
    summary_path = run_dir / "summary.json"
    if not metrics_path.is_file() or not summary_path.is_file():
        print("Incomplete result:", run_dir)
        continue

    metrics = pd.read_csv(metrics_path)
    if "kernel_family" in metrics:
        metrics = metrics[metrics["kernel_family"] == "all"]
    summary = json.loads(summary_path.read_text())
    for split in ("test", "exemplar"):
        selected = metrics[metrics["split"] == split]
        rows.append(
            {
                "experiment": config["experiment_name"],
                "split": split,
                "macro_smape": selected["smape"].mean(),
                "macro_r2": selected["r2"].mean(),
                "best_epoch": summary.get("best_epoch"),
                "best_validation_smape": summary.get("best_metric"),
            }
        )

comparison = pd.DataFrame(rows)
display(comparison)

archive = package_results()
from IPython.display import FileLink
display(FileLink(str(archive)))


## Interpretation guardrails

- Compare the encoders primarily on validation/test SMAPE and their learning
  curves; exemplar SMAPE is a difficult extrapolation diagnostic, not the sole
  selection criterion.
- A small difference at one seed remains inconclusive. The strongest useful
  outcome is a clear accuracy gap, or similar accuracy with a substantial
  throughput/parameter advantage for the relational model.
- A wall-time timeout is right-censoring, not convergence. Check `best_epoch`
  and the tail of `training_history` before treating a capped run as final.
- If the relational model is competitive, repeat the matched pair over multiple
  seeds before adding hierarchical block modeling or broader hyperparameter
  searches.
